In [7]:
# %%
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)
from sklearn.model_selection import RandomizedSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from scipy.stats import loguniform, uniform
from joblib import dump

# add project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features



# ============================================================
# 1. LOAD DATASET AND FILTER
# ============================================================

df_feats, feature_cols = get_features("../data/raw")

df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008) &
    (df_feats["season_end_year"] <= 2024) &
    (df_feats["minutes_played"] >= 100)
].copy()

print("ML Dataset:", df_ml.shape)

ML Dataset: (23442, 79)


In [8]:
final_features = [
    'a_per90_z_lag1', 'ga_per90_z_lag1', 'matches_played_z_lag1',
    'g_per90_z_lag1', 'pen_share_z_lag1', 'g_per90_w', 'ga_per90_w',
    'a_per90_w', 'pen_share_w', 'a_per90_z_delta', 'ga_per90_z_delta',
    'main_position', 'g_per90_z_delta', 'age', 'win_rate', 'height',
    'goals_per_game', 'minutes_played_z_lag1', 'pen_share_z_delta',
    'age_norm', 'team_ucl_strength', 'age_penalty',
    'matches_played_z_delta', 'gc_per90_z_lag1',
    'minutes_played_z_delta', 'season_end_year',
    'Titles', 'num_trophies', 'won_champions',
    'player_id', 'player_name'
]

metadata_cols = [
    "player_id", "player_name", "season_end_year", "minutes_played"
]

final_features = list(dict.fromkeys(final_features))
final_features = [c for c in final_features if c in df_ml.columns]

model_feature_cols = [
    c for c in final_features if c not in metadata_cols
]

print("Model Features:", len(model_feature_cols))

Model Features: 28


In [9]:

df_train = df_ml[df_ml["season_end_year"] <= 2018].copy()
df_val   = df_ml[(df_ml["season_end_year"] >= 2019) &
                 (df_ml["season_end_year"] <= 2022)].copy()

X_train = df_train[model_feature_cols]
y_train = df_train["ballon_dor_winner"].astype(int)

X_val = df_val[model_feature_cols]
y_val = df_val["ballon_dor_winner"].astype(int)

print("Train balance:\n", y_train.value_counts())


Train balance:
 ballon_dor_winner
0    14290
1       11
Name: count, dtype: int64


In [10]:
numeric_features = [c for c in model_feature_cols if df_ml[c].dtype != "object"]
categorical_features = [c for c in model_feature_cols if df_ml[c].dtype == "object"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop"
)


In [11]:

lr = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    max_iter=5000,
    n_jobs=-1
)

pipeline = ImbPipeline(steps=[
    ("preprocess", preprocessor),
    ("smote", SMOTE(k_neighbors=1, random_state=42)),
    ("lr", lr)
])


In [12]:

param_dist = {
    "lr__C": loguniform(1e-3, 10),     # regularization strength
    "lr__l1_ratio": uniform(0, 1),     # elastic-net mixing
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=50,
    scoring="roc_auc",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("\n💙 Best CV AUC:", random_search.best_score_)
print("🏆 Best Params:", random_search.best_params_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits

💙 Best CV AUC: 0.995789840432978
🏆 Best Params: {'lr__C': np.float64(2.0651425578959257), 'lr__l1_ratio': np.float64(0.3567533266935893)}


In [13]:

best_pipeline = random_search.best_estimator_

proba_val = best_pipeline.predict_proba(X_val)[:, 1]
pred_val_default = (proba_val >= 0.5).astype(int)

print("\n=== VALIDATION METRICS ===")
print("AUC:", roc_auc_score(y_val, proba_val))
print("Recall:", recall_score(y_val, pred_val_default))
print("Precision:", precision_score(y_val, pred_val_default))
print("F1:", f1_score(y_val, pred_val_default))
print("Confusion matrix:\n", confusion_matrix(y_val, pred_val_default))


=== VALIDATION METRICS ===
AUC: 0.9997790665562
Recall: 1.0
Precision: 0.3
F1: 0.46153846153846156
Confusion matrix:
 [[6028    7]
 [   0    3]]


In [14]:

from sklearn.metrics import precision_recall_curve

prec, rec, th = precision_recall_curve(y_val, proba_val)

thr_df = pd.DataFrame({
    "threshold": th,
    "precision": prec[:-1],
    "recall": rec[:-1],
    "f1": 2*(prec[:-1]*rec[:-1])/(prec[:-1]+rec[:-1])
})

best_thr = float(thr_df.sort_values("f1", ascending=False).iloc[0]["threshold"])

print("\n⭐ Best threshold by F1:", best_thr)



⭐ Best threshold by F1: 0.9990214103801881
